In [ ]:
#@title Estilo de la clase (ejecutar, no hace falta leer) {display-mode: "form"}
from IPython.display import HTML, display

display(HTML(r"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Work+Sans:wght@400;600&family=Amiri:wght@400;700&display=swap');
.rendered_html, .markdown, .cell .text_cell_render { font-family:'Work Sans',system-ui,sans-serif; color:#122535; }
.rendered_html h1,.rendered_html h2,.rendered_html h3 { font-family:'Amiri',Georgia,serif; color:#00529B; }
.rendered_html h2 { border-bottom:2px solid #00529B; padding-bottom:.2em; }
.rendered_html a { color:#00529B; }
.rendered_html table th { background:#00529B; color:#fff; }
.caja { background:#f3f5f7; border-left:4px solid #00529B; padding:.7em 1em; border-radius:4px; }
.ojo { background:#fdf3ec; border-left:4px solid #C8651B; padding:.7em 1em; border-radius:4px; }
</style>
"""))

# Colores de la clase: PC1 en verde y PC2 en violeta, como en las slides.
VERDE = "#31A354"
VIOLETA = "#756BB1"
AZUL = "#00529B"
GRAFITO = "#4A5B6B"

# Clase 10 · Clustering

**Analítica de Datos** · Maestría en Ciencias del Comportamiento · Universidad de San Andrés

**Primavera 2026 · 10/10/2026**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomdamelio/analitica_de_datos_alumnos/blob/main/clases/clase-10/notebooks/clase10_python.ipynb)

---

La clase pasada resumimos muchas medidas en pocas componentes. Hoy buscamos **grupos de empleados** que trabajan de forma parecida, sin que nadie nos diga de
antemano cuáles son.

Al terminar esta notebook vas a poder:

- agrupar con **K-means** y ver los grupos,
- ver qué cambia con **distintos K** y elegir K con el **método del codo**,
- agrupar con **clustering jerárquico**,
- armar un **dendrograma** y elegir a mano la altura donde cortarlo.

## Contenido

| # | Tema | La idea en una línea |
|---|---|---|
| 1 | [Nueve medidas de cómo trabaja cada empleado](#sec-datos) | los datos de hoy |
| 2 | [Mirar los datos con PCA](#sec-pca) | nueve medidas no se grafican; dos componentes sí |
| 3 | [K-means](#sec-kmeans) | K grupos, distintos K y el codo |
| 4 | [Clustering jerárquico](#sec-jerarquico) | un árbol que se corta donde uno elige |
| 5 | [Cierre](#sec-cierre) | lo que te llevás |

## 1. Nueve medidas de cómo trabaja cada empleado

<a id="sec-datos"></a>

Nimbus registra, para cada uno de sus 600 empleados, nueve medidas de cómo trabaja: seis salen de
las herramientas de trabajo (horas, reuniones, mensajes, contactos, capacitación) y tres de una
encuesta (compromiso, agotamiento y si recomendaría la empresa). Están en `nimbus_patrones.csv`.
Con ellas vamos a buscar **grupos de empleados que trabajan parecido**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 50

BASE = "https://raw.githubusercontent.com/tomdamelio/analitica_de_datos_alumnos/main/data/toy-nimbus/"

patrones = pd.read_csv(BASE + "nimbus_patrones.csv")

datos = patrones

MEDIDAS = [
    "horas_semana",
    "indice_compromiso",
    "horas_fuera_horario_semana",
    "minutos_reunion_semana",
    "mensajes_dia",
    "contactos_distintos_mes",
    "horas_capacitacion_mes",
    "agotamiento",
    "recomendaria_0a10",
]

X = datos[MEDIDAS]

print(X.shape[0], "empleados y", X.shape[1], "medidas")
X.head()

Mirá el promedio y el desvío (`std`) de cada medida:

In [ ]:
X.describe().round(1)

Las medidas están en **unidades muy distintas**. Los minutos de reunión varían en cientos, pero el
agotamiento en una escala de 1 a 7. K-means mide **distancias** entre empleados, entonces la medida
con números más grandes dominaría todo. Por eso, igual que con PCA en la clase pasada, primero
**estandarizamos**: a cada medida le restamos su promedio y la dividimos por su desvío.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_esc = StandardScaler().fit_transform(X)

pd.DataFrame(X_esc, columns=X.columns).describe().round(1)

## 2. Mirar los datos con PCA

<a id="sec-pca"></a>

Nueve medidas son nueve dimensiones, y eso no se puede dibujar. Para **mirar** usamos lo de la
clase pasada: PCA con dos componentes.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pcs = pca.fit_transform(X_esc)

print("varianza explicada por PC1 y PC2:", pca.explained_variance_ratio_.round(3))

fig, ax = plt.subplots(dpi=150)
ax.scatter(pcs[:, 0], pcs[:, 1], s=8, color="gray", alpha=0.6)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Los 600 empleados en PC1 y PC2")

Entre las dos componentes resumen cerca del 63 % de la variación de las nueve medidas, y a ojo se
ven **cuatro nubes**. ¿Lo confirma un método? Para eso está el clustering.

## 3. K-means

<a id="sec-kmeans"></a>

`KMeans` se usa como cualquier modelo de sklearn. Se crea, se ajusta con `.fit()` y se leen los
resultados desde el mismo objeto. Tiene tres argumentos que importan:

- `n_clusters`: la cantidad de grupos, **K**. La elegimos nosotros, como la K de KNN;
- `n_init`: cuántas veces corre el algoritmo con arranques distintos (se queda con la mejor corrida);
- `random_state`: la semilla, para que el resultado sea siempre el mismo.

Probemos con dos grupos:

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=2, n_init=10, random_state=SEED)
km.fit(X_esc)

grupos = km.labels_

print("empleados en cada grupo:")
print(pd.Series(grupos).value_counts())
print()
print("W (inertia_):", km.inertia_)

- `labels_` tiene el grupo de cada empleado: 0 o 1 (binario, le pedimos 2 clusters). El número no significa nada, es solo un nombre.
- `inertia_` es **W**, la variación dentro de los grupos: la suma de las distancias al cuadrado de
  cada empleado al **centroide** (el punto promedio) de su grupo. Cuanto más chica, más apretados
  los grupos.

Pintamos a cada empleado con su grupo, en el plano de PCA:

In [ ]:
fig, ax = plt.subplots(dpi=150)
ax.scatter(pcs[:, 0], pcs[:, 1], c=grupos, cmap="tab10", s=8)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("K-means con K = 2")
plt.show()

### ✏️ Consigna 1: ¿qué pasa con distintos K?

K-means siempre devuelve los K grupos que le pedís, haya o no haya K grupos en los datos. Probá
con K = 2, 3, 4 y 6, y dibujá cada resultado en el plano de PCA, uno al lado del otro.

Pistas:

- adentro del `for`, `KMeans(n_clusters=k, ...)` crea el modelo con la K de esa vuelta;
- los grupos están en `km.labels_`; pasáselos al gráfico en `c=`.

In [ ]:
ks_prueba = [2, 3, 4, 6]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), dpi=150, sharex=True, sharey=True)

for i, k in enumerate(ks_prueba):
    # TODO: K-means con k grupos
    km = KMeans(n_clusters=____, n_init=10, random_state=SEED)
    km.fit(X_esc)

    ax = axes[i]
    # TODO: pintá cada punto con su grupo
    ax.scatter(pcs[:, 0], pcs[:, 1], c=____, cmap="tab10", s=6)
    ax.set_title("K = " + str(k) + "  (W = " + str(round(km.inertia_)) + ")")
    ax.set_xlabel("PC1")

axes[0].set_ylabel("PC2")
plt.show()

Con K = 2 y K = 3, K-means junta nubes que a ojo están separadas. Con K = 4, cada nube es un grupo.
Con K = 6 parte nubes que estaban enteras: inventa grupos porque se los pedimos. Y W (en el título
de cada panel) **baja siempre** que agregamos grupos, así que el K con menor W no sirve para elegir.
Para eso está el codo.

### ✏️ Consigna 2: elegir K con el codo

Corré K-means para K de 1 a 8, guardá el W de cada uno y graficalo. Elegí K donde la curva **deja
de bajar "significativamente"**. Si te fijás, parece como un codo.

Pistas:

- adentro del `for`, `KMeans(n_clusters=k, ...)` crea el modelo con la K de esa vuelta;
- después de `.fit()`, el W está en `.inertia_`; agregalo a la lista con `w.append(...)`.

In [ ]:
ks = [1, 2, 3, 4, 5, 6, 7, 8]
w = []

for k in ks:
    # TODO: K-means con k grupos
    km = KMeans(n_clusters=___, n_init=10, random_state=SEED)
    km.fit(X_esc)
    # TODO: guardá el W de esta corrida
    w.append(___)

fig, ax = plt.subplots(dpi=150)
ax.plot(ks, w, marker="o")
ax.set_xlabel("K, cantidad de grupos")
ax.set_ylabel("W (inertia_)")
ax.set_title("El codo")
plt.show()

W siempre baja al agregar grupos (con 600 grupos, uno por empleado, sería cero), así que no se
elige el K con menor W.

Se mira cuánto baja. Hasta **K = 4** baja mucho, y desde ahí casi nada.
El codo está en 4. Es una decisión a ojo, sí. Pero el libro lo admite, y es lo que se suele hacer.

### Los cuatro grupos sobre PCA

Ajustamos K-means con el K que elegimos y pintamos a cada empleado con su grupo:

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=SEED)
km.fit(X_esc)
grupos = km.labels_

fig, ax = plt.subplots(dpi=150)
ax.scatter(pcs[:, 0], pcs[:, 1], c=grupos, cmap="tab10", s=8)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("K-means con K = 4")

Los cuatro grupos de K-means coinciden con las cuatro nubes que se veían a ojo. Fijate que K-means
**no usó** PC1 ni PC2: agrupó con las nueve medidas, y PCA solo nos dejó verlo.

## 4. Clustering jerárquico

<a id="sec-jerarquico"></a>

El jerárquico no pide K de antemano. Arranca con cada empleado como un grupo solo, junta los dos
más parecidos, y repite hasta que quedan todos juntos. Eso arma un árbol, el **dendrograma**. Usamos dos funciones de `scipy`:

- `linkage(datos, method=...)` hace todas las fusiones. `method` es el **linkage**: cómo se mide la
  distancia entre dos grupos. Usamos `"ward"`, que junta los grupos que menos aumentan W;
- `dendrogram(...)` lo dibuja. Con 600 hojas no se lee nada, así que mostramos solo las 30 ramas de
  arriba (`truncate_mode="lastp", p=30`); el número entre paréntesis abajo de cada rama es cuántos
  empleados tiene.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

Z = linkage(X_esc, method="ward")

fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
dendrogram(Z, truncate_mode="lastp", p=30, ax=ax, color_threshold=0, above_threshold_color="gray")
ax.set_ylabel("altura de fusión")
ax.set_title("Dendrograma (Ward), 600 empleados")

Se lee por la **altura**: cuanto más abajo se unen dos ramas, más parecidos son sus empleados. Que
dos hojas estén al lado en el eje horizontal no dice nada.

Para formar grupos se **corta** el árbol con una línea horizontal: cada rama que cuelga por debajo
de la línea es un grupo. Mové la altura y mirá cuántos grupos quedan:

In [ ]:
from ipywidgets import FloatSlider, interact

def cortar(altura):
    fig, ax = plt.subplots(figsize=(9, 4), dpi=110)
    dendrogram(Z, truncate_mode="lastp", p=30, ax=ax, color_threshold=altura, above_threshold_color="gray")
    ax.axhline(altura, color="black", linestyle="--")
    ax.set_ylabel("altura de fusión")
    grupos_corte = fcluster(Z, t=altura, criterion="distance")
    ax.set_title("Cortando a altura " + str(round(altura, 1)) + ": " + str(len(set(grupos_corte))) + " grupos")
    plt.show()

interact(cortar, altura=FloatSlider(value=25, min=5, max=65, step=1));

Cortando entre 13 y 34, quedan cuatro grupos. Entre 34 y 44, tres. Más arriba, dos. Dónde cortar
es, como casi siempre en machine learning, una decisión nuestra.

### ✏️ Consigna 3: elegir dónde cortar

Elegí una altura de corte mirando el dendrograma, dibujá la línea, cortá el árbol ahí y pintá los
grupos que quedan en el plano de PCA.

Pistas:

- `ax.axhline(altura)` dibuja la línea horizontal;
- `fcluster(Z, t=altura, criterion="distance")` devuelve el grupo de cada empleado si se corta a esa
  altura (los grupos se numeran desde 1);
- el gráfico de PCA es el de K-means, con `c=grupos_arbol`.

In [ ]:
# TODO: la altura que elegiste mirando el dendrograma
altura = ___

fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
dendrogram(Z, truncate_mode="lastp", p=30, ax=ax, color_threshold=altura, above_threshold_color="gray")
ax.axhline(altura, color="black", linestyle="--")
ax.set_ylabel("altura de fusión")
ax.set_title("Corte a mano")
plt.show()

# TODO: cortá el árbol a esa altura
grupos_arbol = fcluster(Z, t=___, criterion="distance")

print("empleados por grupo del árbol:")
print(pd.Series(grupos_arbol).value_counts().sort_index())

fig, ax = plt.subplots(dpi=150)
# TODO: pintá con los grupos del árbol
ax.scatter(pcs[:, 0], pcs[:, 1], c=___, cmap="tab10", s=8)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Grupos del árbol, cortando a " + str(altura))
plt.show()

Cortando a 25 quedan cuatro grupos, y en el plano de PCA son las mismas cuatro nubes que encontró
K-means, salvo unos pocos empleados en el borde entre dos nubes. Dos métodos distintos llegan casi
al mismo lugar (los números de grupo no coinciden, porque son solo nombres). Si cortás más arriba,
el árbol junta nubes, igual que K-means con K = 2 o 3.

## 5. Cierre

<a id="sec-cierre"></a>

### Lo que te llevás

1. **Clustering agrupa personas, no columnas.** Cada empleado cae en un solo grupo.
2. **K-means busca el W más chico** para la K que le pedimos, y siempre devuelve esa cantidad de
   grupos, existan o no.
3. **K es una decisión.** W baja siempre; el codo ayuda, pero es a ojo.
4. **El dendrograma se lee por la altura** y se corta donde uno elige: cada altura es una cantidad
   de grupos.
5. **Escalar antes de agrupar**, porque todo lo que mide distancias depende de las unidades.

### Todo el código de hoy, en pocas líneas

```python
X_esc = StandardScaler().fit_transform(X)                     # escalar
pcs = PCA(n_components=2).fit_transform(X_esc)                # para mirar
km = KMeans(n_clusters=4, n_init=10, random_state=SEED)       # K-means
km.fit(X_esc)
grupos = km.labels_                                           # el grupo de cada empleado
w = km.inertia_                                               # W, para el codo
Z = linkage(X_esc, method="ward")                             # jerárquico
grupos_arbol = fcluster(Z, t=25, criterion="distance")        # cortar a mano
```

### Para seguir

- James y otros, capítulo 12, sección 4, es la lectura de esta clase.
- La clase que viene: **texto**. Modelos de lenguaje, embeddings y sentimiento.